In [ ]:
# Databricks notebook sourceMAGIC %md## Purpose: Centralize enterprise patterns for Silver: quarantining, dedupe, normalization, date fixes, integrity checks.These helpers are used by all domain notebooks.## Why these utilities?Quarantine: Enterprise must never silently drop bad data—everything traceable.Dedup: Latest record wins per business key (common Silver rule).Normalize categories: Avoid downstream “M vs male vs Male” breakage.Date sanity: Production data often has reversed dates; fix safely.Nulls: Clear, predictable imputations.FK checks: Enforce referential integrity Claims↔Policies↔Members↔Providers, quarantine orphans.

In [ ]:
MAGIC %run /Workspace/Repos/karthikvanapalli1505@gmail.com/Bupa_Insuarnce_Project/00_Pre_Pilot/_00__Pre_Connectors

In [ ]:
# ===============================================# _00_utils_silver.py (Enterprise-level utilities)# ===============================================from pyspark.sql import functions as Ffrom pyspark.sql.types import *from pyspark.sql import DataFrame, Window as Wfrom datetime import datetime# --- Configuration (edit once) ---ACCOUNT = "bupaprocesseddatastorage"CATALOG_SILVER = "bupa_silver"CONTAINER_SILVER = "silver"CONTAINER_BRONZE = "bronze"# Paths (Delta locations)paths_bronze = {    "policies":  f"abfss://{CONTAINER_BRONZE}@{ACCOUNT}.dfs.core.windows.net/delta_tables/policies",    "members":   f"abfss://{CONTAINER_BRONZE}@{ACCOUNT}.dfs.core.windows.net/delta_tables/members",    "claims":    f"abfss://{CONTAINER_BRONZE}@{ACCOUNT}.dfs.core.windows.net/delta_tables/claims",    "providers": f"abfss://{CONTAINER_BRONZE}@{ACCOUNT}.dfs.core.windows.net/delta_tables/providers"}paths_silver = {    "policies":  f"abfss://{CONTAINER_SILVER}@{ACCOUNT}.dfs.core.windows.net/policies",    "members":   f"abfss://{CONTAINER_SILVER}@{ACCOUNT}.dfs.core.windows.net/members",    "claims":    f"abfss://{CONTAINER_SILVER}@{ACCOUNT}.dfs.core.windows.net/claims",    "providers": f"abfss://{CONTAINER_SILVER}@{ACCOUNT}.dfs.core.windows.net/providers",    "_quarantine": f"abfss://{CONTAINER_SILVER}@{ACCOUNT}.dfs.core.windows.net/_quarantine"}spark.sql(f"CREATE DATABASE IF NOT EXISTS {CATALOG_SILVER}")# -----------------------------# 1. Data Quality Functions# -----------------------------def null_percentage(df: DataFrame):    total = df.count()    return df.select([        (F.count(F.when(F.col(c).isNull(), c)) / total).alias(c)        for c in df.columns    ])def check_duplicates(df: DataFrame, key_cols: list):    dup_count = df.groupBy(key_cols).count().filter("count > 1").count()    if dup_count > 0:        print(f"⚠️ Found {dup_count} duplicate rows for keys {key_cols}")    else:        print("✅ No duplicate keys found.")    return df.dropDuplicates(key_cols)def enforce_schema(df: DataFrame, target_schema: StructType):    for field in target_schema:        df = df.withColumn(field.name, F.col(field.name).cast(field.dataType))    return dfdef data_quality_summary(df: DataFrame, table_name: str):    print(f"\n===== Data Quality Report for {table_name} =====")    df.printSchema()    print("Row Count:", df.count())    print("Null Percentage:")    display(null_percentage(df))# -----------------------------# 2. Metadata Tracking# -----------------------------def add_audit_columns(df: DataFrame, source_name: str):    return (df        .withColumn("created_at", F.lit(datetime.now()))        .withColumn("source_system", F.lit(source_name))        .withColumn("record_hash", F.sha2(F.concat_ws("||", *df.columns), 256))    )# ---------- Quarantine ----------"""def quarantine(df: DataFrame, reason: str, table_name: str):    qpath = f"{paths_silver['_quarantine']}/{table_name}"    (df.withColumn("_reject_reason", F.lit(reason))       .withColumn("_reject_ts", F.current_timestamp())       .write.format("delta").mode("append").save(qpath))"""# ---------- Quarantine (new enterprise-safe version) ----------from pyspark.sql import functions as Fdef quarantine(df, code: str, table_label: str):    """    Stable quarantine sink: envelope columns + JSON payload.    Works regardless of the dataframe's schema.    """    if df is None:        return    qt = (df          .withColumn("_quarantine_code",  F.lit(code))          .withColumn("_quarantine_table", F.lit(table_label))          .withColumn("_quarantine_ts",    F.current_timestamp())          .withColumn("payload",           F.to_json(F.struct(*[F.col(c) for c in df.columns])))          .select("_quarantine_code","_quarantine_table","_quarantine_ts","payload"))    (qt.write       .format("delta")       .mode("append")       .saveAsTable(f"{DB_SILVER}.__quarantine"))    print(f"[QUARANTINE] {code} → {DB_SILVER}.__quarantine (rows={qt.count()})")# ---------- Dedupe ----------def drop_dupes_keep_latest(df: DataFrame, key_cols: list, order_desc_cols: list) -> DataFrame:    w = W.partitionBy(*key_cols).orderBy(*[F.desc(c) for c in order_desc_cols])    return (df.withColumn("_rn", F.row_number().over(w))              .filter(F.col("_rn") == 1)              .drop("_rn"))# ---------- Categorical normalization ----------def normalize_categories(df: DataFrame) -> DataFrame:    d = df    if "Gender" in d.columns:        d = d.withColumn("Gender",                         F.when(F.lower(F.col("Gender")).isin("m","male"), "M")                          .when(F.lower(F.col("Gender")).isin("f","female"), "F")                          .otherwise(F.col("Gender")))    if "Smoker" in d.columns:        d = d.withColumn("Smoker",                         F.when(F.lower(F.col("Smoker")).isin("y","yes","1"), "Y")                          .when(F.lower(F.col("Smoker")).isin("n","no","0"), "N")                          .otherwise(F.col("Smoker")))    return d# ---------- Date sanity ----------def fix_dates(df: DataFrame, start_col: str, end_col: str) -> DataFrame:    return (df      .withColumn("_start", F.col(start_col))      .withColumn("_end",   F.col(end_col))      .withColumn("_swap",  (F.col("_end") < F.col("_start")))      .withColumn(start_col, F.when(F.col("_swap"), F.col("_end")).otherwise(F.col("_start")))      .withColumn(end_col,   F.when(F.col("_swap"), F.col("_start")).otherwise(F.col("_end")))      .drop("_start","_end","_swap"))# ---------- Null handling helpers ----------def impute_zero(df: DataFrame, colname: str) -> DataFrame:    return df.withColumn(colname, F.when(F.col(colname).isNull(), F.lit(0.0)).otherwise(F.col(colname)))def fill_with_mode(df: DataFrame, colname: str) -> DataFrame:    mode_val = df.groupBy(colname).count().orderBy(F.desc("count")).first()    if mode_val and mode_val[0] is not None:        return df.fillna({colname: mode_val[0]})    return df# ---------- FK validation ----------def fk_filter(df: DataFrame, fk_col: str, ref_df: DataFrame, ref_key: str, table_name: str, reason: str):    joined = df.join(ref_df.select(F.col(ref_key).alias("_ref")), df[fk_col] == F.col("_ref"), "left")    bad = joined.filter(F.col("_ref").isNull())    good = joined.filter(F.col("_ref").isNotNull()).drop("_ref")    if bad.take(1):        quarantine(bad.drop("_ref"), reason, table_name)    return goodprint("Silver utils ready.")

In [ ]:
# =================== DQ UTILITIES (drop-in) ===================from pyspark.sql import functions as Fdef dq_expect(df, name: str, expr: str, severity: str, table_label: str, quarantine_fn, quarantine_code: str):    """    Evaluate a boolean SQL expression on the DataFrame.    - severity in {"error","warn"} controls whether to raise Exception.    - quarantine_fn(df_bad, code, table_label) is called when any violations.    """    total = df.count()    bad  = df.filter(f"NOT ({expr})")    bad_cnt = bad.count()    if bad_cnt > 0:        frac = round(bad_cnt / max(total,1), 6)        print(f"❌ DQ FAIL [{table_label}] {name}: {bad_cnt}/{total} rows ({frac*100:.4f}%) violate: {expr}")        if quarantine_fn and bad_cnt > 0:            quarantine_fn(bad, quarantine_code, table_label)        if severity.lower() == "error":            raise Exception(f"DQ gate failed: {table_label} · {name}")    else:        print(f"✅ DQ PASS [{table_label}] {name}")# ---------- DQ Reference Validation ----------def dq_left_anti_ref(df, df_ref, key_col: str, ref_col: str, name: str, severity: str,                     table_label: str, quarantine_fn, quarantine_code: str):    """    Validate that all df[key_col] exist in df_ref[ref_col].    Uses left_anti join to avoid duplicate columns.    """    ref = df_ref.select(F.col(ref_col).alias("_ref_key")).dropDuplicates()    bad = df.join(ref, F.col(key_col) == F.col("_ref_key"), "left_anti")    bad_cnt = bad.count()    total = df.count()    if bad_cnt > 0:        pct = (bad_cnt / max(total, 1)) * 100.0        print(f"❌ DQ FAIL [{table_label}] {name}: {bad_cnt}/{total} rows ({pct:.4f}%) missing in reference")        if quarantine_fn:            quarantine_fn(bad, quarantine_code, table_label)        if severity.lower() == "error":            raise Exception(f"DQ gate failed: {table_label} · {name}")    else:        print(f"✅ DQ PASS [{table_label}] {name}")def optimize_zorder(full_table_name: str, zcols: list[str]):    from pyspark.sql import SparkSession    spark = SparkSession.getActiveSession()    if not zcols:        spark.sql(f"OPTIMIZE {full_table_name}")    else:        cols = ", ".join([f"`{c}`" for c in zcols])        spark.sql(f"OPTIMIZE {full_table_name} ZORDER BY ({cols})")    # Optional hygiene:    # spark.sql(f"VACUUM {full_table_name} RETAIN 168 HOURS")  # 7 days; align with org policy# ======================================================# CENTRALIZED DQ SLA THRESHOLDS# ======================================================DQ_SLA = {    "fk_members_policy": 0.5,     # % orphans tolerated    "region_enum": 0.0,           # once governed, should be 0    "bmi_warn": 1.0,              # acceptable warn-level outliers    "discount_warn": 0.5}

In [ ]:
DB_SILVER = "bupa_silver"  # adjust if needed# Park the old table (keeps your historical quarantines)spark.sql(f"DROP TABLE IF EXISTS {DB_SILVER}.__quarantine_legacy")try:    spark.sql(f"ALTER TABLE {DB_SILVER}.__quarantine RENAME TO {DB_SILVER}.__quarantine_legacy")except Exception as e:    print("Rename skipped (likely didn't exist):", e)# Create the new stable quarantine tablespark.sql(f"DROP TABLE IF EXISTS {DB_SILVER}.__quarantine")spark.sql(f"""CREATE TABLE {DB_SILVER}.__quarantine (  _quarantine_code   STRING,  _quarantine_table  STRING,  _quarantine_ts     TIMESTAMP,  payload            STRING) USING DELTA""")

In [ ]:
def write_silver(df, full_table, key_cols=None, merge_ts_col=None, mode="overwrite"):    if mode == "merge" and key_cols and merge_ts_col:        spark.sql(f"CREATE TABLE IF NOT EXISTS {full_table} USING DELTA AS SELECT * FROM (SELECT NULL AS __init__) WHERE 1=0")        tgt = spark.table(full_table)        cond = " AND ".join([f"s.{k}=t.{k}" for k in key_cols])        (df.alias("s").write           .format("delta")           .mode("overwrite")    # write to temp location or use DeltaTable for direct merge           .saveAsTable(f"{full_table}__staging"))        spark.sql(f"""        MERGE INTO {full_table} t        USING {full_table}__staging s        ON {cond}        WHEN MATCHED AND s.{merge_ts_col} >= t.{merge_ts_col} THEN UPDATE SET *        WHEN NOT MATCHED THEN INSERT *        """)        spark.sql(f"DROP TABLE IF EXISTS {full_table}__staging")    else:        (df.write.format("delta")            .mode("overwrite")            .option("overwriteSchema", "true")            .saveAsTable(full_table))        # ======================================================# ENHANCED WRITE_SILVER (supports overwrite or merge)# ======================================================from delta.tables import DeltaTabledef write_silver(df, full_table, key_cols=None, merge_ts_col=None, mode="overwrite"):    if mode == "merge" and key_cols and merge_ts_col:        # ensure target table exists        spark.sql(f"CREATE TABLE IF NOT EXISTS {full_table} USING DELTA AS SELECT * FROM (SELECT NULL AS __init__) WHERE 1=0")        tgt = DeltaTable.forName(spark, full_table)        cond = " AND ".join([f"s.{k}=t.{k}" for k in key_cols])        (tgt.alias("t")            .merge(df.alias("s"), cond)            .whenMatchedUpdateAll(condition=f"s.{merge_ts_col} >= t.{merge_ts_col}")            .whenNotMatchedInsertAll()            .execute())        print(f"[MERGE] {df.count()} rows merged into {full_table}")    else:        (df.write            .format("delta")            .mode("overwrite")            .option("overwriteSchema", "true")            .saveAsTable(full_table))        print(f"[WRITE] Overwrote {full_table} (schema updated)")